In [1]:
import networkx as nx
import pandas as pd
from pyvis.network import Network

In [2]:
df = pd.read_csv('../data/raw/facebook_combined.txt', sep=' ', names=['first_node', 'last_node'])
G = nx.Graph()
G.add_edges_from(df.values)

In [3]:
comms_list = nx.community.louvain_communities(G)
node_comms = {}

for index, com in enumerate(comms_list):
    for n in com:
        node_comms[n] = index

In [4]:
#nx.set_node_attributes(G, node_comms, 'community')

In [5]:
import numpy as np

for n, data in G.nodes(data=True):
    for k, v in data.items():
        if isinstance(v, (np.integer, np.int64)):
            data[k] = int(v)
        elif isinstance(v, (np.floating, np.float64)):
            data[k] = float(v)
        elif isinstance(v, (set, list, dict)):
            data[k] = str(v)  # PyVis cannot handle these

In [9]:
# --- 1. Create a clean graph with string node IDs ---
H = nx.Graph()

# Convert node IDs to strings
mapping = {n: str(n) for n in G.nodes()}
H = nx.relabel_nodes(G, mapping, copy=True)

# Remove attributes (PyVis hates them)
for n in H.nodes():
    H.nodes[n].clear()

# --- 2. Create PyVis network ---
net = Network(
    height="900px",
    width="100%",
    bgcolor="#ffffff",
    font_color="black",
    notebook=False
)

net.from_nx(H)

# --- 3. Add community colours ---
for node in net.nodes:
    original_id = node["id"]
    # Convert back to original type for lookup
    original = list(mapping.keys())[list(mapping.values()).index(original_id)]
    community = G.nodes[original].get("community", 0)
    node["color"] = f"hsl({(community * 47) % 360}, 80%, 50%)"
    node["size"] = 8

# --- 4. Show graph ---
net.write_html("network_view.html")